# 3. Mesh postprocessing for real-world recognition

The mesh generated by DreamGrasp is produced based on Zero123's camera distance of 3.8.  
To adapt it to the actual camera distance, we simply rescale the output mesh.

In [1]:
import numpy as np
import open3d as o3d
import os
import pickle

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


The data we generated in PyBullet uses a camera distance of 1.5 m.

In [2]:
gt_camera_distance=1.5

In [3]:
def postprocess_predicted_mesh(mesh, gt_camera_distance, dreamgrasp_camera_distance=3.8):
	mesh.scale(gt_camera_distance/dreamgrasp_camera_distance, np.array([0, 0, 0]))
	mesh.paint_uniform_color([1., 0., 0.])
	return mesh  

Load the DreamGrasp outputs after running `python launch.py --config custom/DreamGrasp/configs/dreamgrasp.yaml --train --gpu 0`

In [ ]:
predicted_meshes = [
    postprocess_predicted_mesh(o3d.io.read_triangle_mesh("../threestudio/outputs/dreamgrasp/exp/save/it3000-tsdf_mesh/model_0.obj"), gt_camera_distance),
    postprocess_predicted_mesh(o3d.io.read_triangle_mesh("../threestudio/outputs/dreamgrasp/exp/save/it3000-tsdf_mesh/model_1.obj"), gt_camera_distance)
]

Compare with the ground-truth mesh

In [5]:
data_path = '../datasets/Objects_2/0'
with open(os.path.join(data_path, 'data.pkl'), "rb") as f: # you can change this directory
    data = pickle.load(f)

gt_meshes = [o3d.io.read_triangle_mesh(os.path.join(data_path, p)) for p in os.listdir(data_path) if p.endswith(".obj")]

o3d.visualization.draw_geometries(predicted_meshes + gt_meshes)